# 🌮 Rey Taco Picks — Laboratorio de Medición en Modo Sombra (Shadow Measurement)

**Modo 100% Solo Lectura (No-Intrusivo)**: Este notebook de Google Colab analiza el desempeño real de **Rey Taco Picks** sin modificar ningún archivo de producción (`verified_record.json` ni el bot en vivo).

### Qué mide este notebook:
1. **Rendimiento por Deporte**: Cartera Activa (`NBA`, `NFL`, `Tenis`) vs Histórico Completo.
2. **Closing Line Value (CLV %) Real (`n=140`)**: Ventaja obtenida respecto a la cuota de cierre antes del partido (`+1.83%` global, `+2.79%` en líneas con movimiento, `+3.21%` en NBA).
3. **Calibración Brier Score y Log-Loss Pareados (`n=113`)**: Comparación entre probabilidad desmarginada al momento de entrada vs probabilidad de cierre.
4. **Eficiencia por Banda de Cuota y Simulación Monte Carlo (100,000 trayectorias)**.

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SHADOW_SNAPSHOT = {
  "mode": "SHADOW_MEASUREMENT_READONLY",
  "affects_production": False,
  "active_core_portfolio": {
    "won": 78, "lost": 63, "void": 1, "net_units": 16.98,
    "settled_picks": 141, "win_rate_pct": 55.3, "roi_pct": 12.0
  },
  "by_sport": {
    "nba": {"won": 42, "lost": 35, "net_units": 8.50, "settled_picks": 77, "win_rate_pct": 54.5, "roi_pct": 11.0},
    "nfl": {"won": 12, "lost": 6, "net_units": 5.91, "settled_picks": 18, "win_rate_pct": 66.7, "roi_pct": 32.8},
    "tennis": {"won": 24, "lost": 22, "net_units": 2.57, "settled_picks": 46, "win_rate_pct": 52.2, "roi_pct": 5.6},
    "soccer": {"won": 45, "lost": 75, "net_units": -28.42, "settled_picks": 120, "win_rate_pct": 37.5, "roi_pct": -23.7},
    "baseball": {"won": 9, "lost": 15, "net_units": -7.39, "settled_picks": 24, "win_rate_pct": 37.5, "roi_pct": -30.8}
  },
  "clv_by_sport": {
    "nba": {"samples": 39, "mean_clv_pct": 3.21, "median_clv_pct": 2.17, "moved_lines": 30, "mean_moved_clv_pct": 4.17, "beat_closing_count": 20, "lost_closing_count": 10, "moved_positive_rate_pct": 66.7},
    "baseball": {"samples": 22, "mean_clv_pct": 3.81, "median_clv_pct": 1.06, "moved_lines": 13, "mean_moved_clv_pct": 6.44, "beat_closing_count": 11, "lost_closing_count": 2, "moved_positive_rate_pct": 84.6},
    "tennis": {"samples": 21, "mean_clv_pct": 0.86, "median_clv_pct": 0.0, "moved_lines": 7, "mean_moved_clv_pct": 2.59, "beat_closing_count": 4, "lost_closing_count": 3, "moved_positive_rate_pct": 57.1},
    "soccer": {"samples": 48, "mean_clv_pct": 0.81, "median_clv_pct": 0.0, "moved_lines": 34, "mean_moved_clv_pct": 1.14, "beat_closing_count": 19, "lost_closing_count": 15, "moved_positive_rate_pct": 55.9},
    "nfl": {"samples": 10, "mean_clv_pct": -0.89, "median_clv_pct": -0.17, "moved_lines": 8, "mean_moved_clv_pct": -1.12, "beat_closing_count": 3, "lost_closing_count": 5, "moved_positive_rate_pct": 37.5}
  },
  "paired_brier_and_logloss_calibration": {
    "all_tracked_sports": {"paired_settled_samples": 113, "empirical_win_rate": 0.5133, "entry_devig_brier_score": 0.2519, "closing_devig_brier_score": 0.2502, "closing_vs_entry_brier_delta": -0.0018, "entry_devig_log_loss": 0.6972, "closing_devig_log_loss": 0.6934, "closing_vs_entry_logloss_delta": -0.0038},
    "active_core_portfolio": {"paired_settled_samples": 58, "empirical_win_rate": 0.6379, "entry_devig_brier_score": 0.2629, "closing_devig_brier_score": 0.2569, "closing_vs_entry_brier_delta": -0.0061, "entry_devig_log_loss": 0.7194, "closing_devig_log_loss": 0.7069, "closing_vs_entry_logloss_delta": -0.0124}
  },
  "odds_band_performance": {
    "favorito_fuerte (<1.70)": {"settled_picks": 14, "won": 8, "lost": 6, "net_units": -0.93, "win_rate_pct": 57.1, "implied_breakeven_pct": 61.2, "edge_over_breakeven_pp": -4.04, "roi_pct": -6.6},
    "linea_principal (1.70-1.99)": {"settled_picks": 100, "won": 58, "lost": 42, "net_units": 9.48, "win_rate_pct": 58.0, "implied_breakeven_pct": 53.2, "edge_over_breakeven_pp": 4.77, "roi_pct": 9.5},
    "plus_money (2.00-2.49)": {"settled_picks": 170, "won": 66, "lost": 104, "net_units": -26.37, "win_rate_pct": 38.8, "implied_breakeven_pct": 46.1, "edge_over_breakeven_pp": -7.26, "roi_pct": -15.5},
    "underdog_alto (2.50+)": {"settled_picks": 1, "won": 0, "lost": 1, "net_units": -1.0, "win_rate_pct": 0.0, "implied_breakeven_pct": 40.0, "edge_over_breakeven_pp": -40.0, "roi_pct": -100.0}
  }
}

print("=== 1. DESEMPEÑO MEDIDO POR DEPORTE ===")
display(pd.DataFrame.from_dict(SHADOW_SNAPSHOT["by_sport"], orient="index"))
print("\n=== 2. CLOSING LINE VALUE (CLV %) POR DEPORTE (n=140) ===")
display(pd.DataFrame.from_dict(SHADOW_SNAPSHOT["clv_by_sport"], orient="index"))
print("\n=== 3. BRIER SCORE Y LOG-LOSS PAREADOS (ENTRADA VS CIERRE) ===")
display(pd.DataFrame.from_dict(SHADOW_SNAPSHOT["paired_brier_and_logloss_calibration"], orient="index"))

In [ ]:
df_bands = pd.DataFrame.from_dict(SHADOW_SNAPSHOT["odds_band_performance"], orient="index")
print("=== 4. CALIBRACIÓN POR BANDA DE CUOTA (WIN RATE REAL VS BREAKEVEN IMPLÍCITO) ===")
display(df_bands)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(df_bands))
width = 0.36
ax.bar(x - width/2, df_bands["win_rate_pct"], width, label="Win Rate Real (%)", color="#22c55e")
ax.bar(x + width/2, df_bands["implied_breakeven_pct"], width, label="Breakeven Implícito (%)", color="#64748b")
ax.set_xticks(x)
ax.set_xticklabels(df_bands.index, rotation=15)
ax.set_ylabel("Porcentaje (%)")
ax.set_title("Rey Taco Picks — Win Rate Empírico vs Breakeven por Banda de Cuota")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5. Simulación Monte Carlo (100,000 trayectorias) sobre la Cartera Activa (NBA + NFL + Tenis)
rng = np.random.default_rng(42)
n_sims = 100_000
horizon_picks = 250
p_win = SHADOW_SNAPSHOT["active_core_portfolio"]["win_rate_pct"] / 100.0
avg_odds = 1.0 + (SHADOW_SNAPSHOT["active_core_portfolio"]["net_units"] + SHADOW_SNAPSHOT["active_core_portfolio"]["lost"]) / SHADOW_SNAPSHOT["active_core_portfolio"]["won"]

wins = rng.random((n_sims, horizon_picks)) < p_win
step_pnl = np.where(wins, avg_odds - 1.0, -1.0)
equity_curves = np.cumsum(step_pnl, axis=1)
peaks = np.maximum.accumulate(np.maximum(0.0, equity_curves), axis=1)
drawdowns = peaks - equity_curves
max_drawdowns = np.max(drawdowns, axis=1)
final_units = equity_curves[:, -1]

print(f"Cuota promedio implícita cartera activa: {avg_odds:.3f}")
print(f"Ganancia esperada tras {horizon_picks} picks: {np.mean(final_units):+.2f} U (Mediana: {np.median(final_units):+.2f} U)")
print(f"Probabilidad de terminar en positivo tras {horizon_picks} picks: {np.mean(final_units > 0)*100:.1f}%")
print(f"Max Drawdown Mediano: {np.median(max_drawdowns):.2f} U | Percentil 95: {np.percentile(max_drawdowns, 95):.2f} U")